# Build a Travily and Invoke Gen AI with Bedrock Chatbot

In [1]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langgraph langgraph-prebuilt langchain-tavily
%pip install boto3
%pip install --upgrade python-dotenv

In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

os.environ["AWS_ACCESS_KEY_ID"] = os.getenv('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv('AWS_SECRET_ACCESS_KEY')
os.environ["AWS_DEFAULT_REGION"] =os.getenv('AWS_DEFAULT_REGION')
os.environ["TAVILY_API_KEY"] = os.getenv('TAVILY_API_KEY')
# os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

In [3]:
os.environ["LANGSMITH_API_KEY"] = os.getenv('LANGSMITH_API_KEY')
os.environ["LANGSMITH_ENDPOINT"]="https://api.smith.langchain.com"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "tutorial"

## Define the tool

Define the web search tool:



In [4]:
from langchain_tavily import TavilySearch

search_tool = TavilySearch(max_results=2)

# Define the tool use AWS API Gateway endpoint

In [5]:
from langchain_core.tools import tool

tools = [search_tool]
search_tool.invoke("What's a 'node' in LangGraph?")

{'query': "What's a 'node' in LangGraph?",
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.linkedin.com/pulse/langgraph-nodes-agents-multi-agent-composition-walid-negm-kaxwe',
   'title': '🧠 LangGraph — Nodes, Agents, and Multi- ...',
   'content': 'LangGraph is agnostic to node content: a node is simply a callable (function or class) that consumes and updates graph state. Whether a node is',
   'score': 0.8548001,
   'raw_content': None},
  {'url': 'https://dev.to/raunaklallala/understanding-core-concepts-of-langgraph-deep-dive-1d7h',
   'title': 'Understanding Core Concepts of LangGraph (Deep Dive)',
   'content': 'At the core, LangGraph has three simple but powerful building blocks: **Nodes, Edges, and State**. Each of those is a *Node*. In LangGraph, a Node can be many things:. Each Node is like a worker with a simple contract: it takes an input, does its piece of the job, and pushes out an output. **Analogy:** Nodes are like “station

## Define the graph



In [6]:
from langchain_aws import ChatBedrock

llm = ChatBedrock(
   model_id="amazon.nova-lite-v1:0",
   temperature=1,
)

We can now incorporate it into a StateGraph:

In [18]:
from typing import Annotated

from langgraph.prebuilt import ToolNode
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", "tools")
graph_builder.add_edge("tools", END)

graph = graph_builder.compile()

In [19]:
from langchain_core.messages import AIMessage, HumanMessage
result= graph.invoke({"messages": [HumanMessage(content="who is the president of india?")]})

In [20]:
result

{'messages': [HumanMessage(content='who is the president of india?', additional_kwargs={}, response_metadata={}, id='b5510da1-3145-49f5-8e9a-e95dbc2e49f1'),
  AIMessage(content=[{'type': 'text', 'text': '<thinking>I need to find out who the current president of India is. I should use the Tavily search engine to retrieve the most recent and accurate information.</thinking>\n'}, {'type': 'tool_use', 'name': 'tavily_search', 'input': {'search_depth': 'advanced', 'query': 'who is the president of India?', 'topic': 'news'}, 'id': 'tooluse_P1a1xKcfc090VR3ZSZNMRn'}], additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '624833cc-e1c7-42b4-871d-d916a51a6446', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Fri, 13 Mar 2026 08:04:12 GMT', 'content-type': 'application/json', 'content-length': '552', 'connection': 'keep-alive', 'x-amzn-requestid': '624833cc-e1c7-42b4-871d-d916a51a6446'}, 'RetryAttempts': 0}, 'stopReason': 'tool_use', 'metrics': {'latencyMs': [909]}, 'model_p